# Tutorial: Scopes and Sets Smoke Test

Audience:
- AMDockVS developers.

Prerequisites:
- The temporary project contains a minimal importable inventory.

Learning goals:
- Verify MoleculeScope against project.db.
- Materialize reproducible sets with runtime.molecules.create_set(...).

## Outline

1. Set up
2. Import baseline data
3. Build lazy scopes
4. Materialize explicit sets

In [ ]:
from __future__ import annotations

import sys
import tempfile
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError("Could not locate repo root")


REPO_ROOT = find_repo_root(Path.cwd())
AMDOCK_SRC = REPO_ROOT / "src"
MOLSUITE_SRC = REPO_ROOT.parent / "MolSuite" / "src"
for path in (AMDOCK_SRC, MOLSUITE_SRC):
    text = str(path)
    if text not in sys.path:
        sys.path.insert(0, text)

from amdockvs.runtime import AMDockVSRuntime

TMP_ROOT = REPO_ROOT / "notebooks" / "_tmp"
TMP_ROOT.mkdir(parents=True, exist_ok=True)
PROJECT_DIR = Path(tempfile.mkdtemp(prefix="03_scopes_snapshots_", dir=TMP_ROOT))
LIGAND_FILE = REPO_ROOT / "data" / "docking" / "1iep_ligand.sdf"
RECEPTOR_FILE = REPO_ROOT / "data" / "docking" / "1iep_receptor.pdbqt"

runtime = AMDockVSRuntime()
runtime.create_project(
    name=f"scopes-snapshots-{PROJECT_DIR.name}",
    folder=PROJECT_DIR,
    description="Notebook smoke test for scopes and snapshots.",
)

job_ids = [
    *runtime.loader.load_ligands([LIGAND_FILE], executor_name="process"),
    *runtime.loader.load_receptors([RECEPTOR_FILE], executor_name="process"),
]
status_map = runtime.wait_for_jobs(job_ids, timeout_s=180.0, poll_s=0.2)
assert all(row.status == "completed" for row in status_map.values()), status_map
status_map


In [ ]:
ligand_scope = runtime.molecules.filter(
    runtime.molecules.all(),
    filters={"primary_role": "ligand", "input_format": "sdf"},
)
receptor_scope = runtime.molecules.select(role="receptor")

ligand_ids = list(runtime.molecules.stream_ids(ligand_scope))
receptor_ids = list(runtime.molecules.stream_ids(receptor_scope))

assert ligand_ids, "Expected at least one ligand id"
assert receptor_ids, "Expected at least one receptor id"

{
    "ligand_ids": ligand_ids,
    "receptor_ids": receptor_ids,
}


In [ ]:
ligand_set = runtime.molecules.create_set(ligand_scope, name="sdf_ligands", kind="snapshot")
receptor_set = runtime.molecules.create_set(receptor_scope, name="all_receptors", kind="snapshot")

assert ligand_set.id > 0
assert receptor_set.id > 0

ligand_ids_from_set = list(runtime.molecules.stream_ids(runtime.molecules.all(source=ligand_set)))
receptor_ids_from_set = list(runtime.molecules.stream_ids(runtime.molecules.all(source=receptor_set)))

assert ligand_ids_from_set == ligand_ids
assert receptor_ids_from_set == receptor_ids

{
    "ligand_set_id": ligand_set.id,
    "receptor_set_id": receptor_set.id,
    "ligand_ids_from_set": ligand_ids_from_set,
    "receptor_ids_from_set": receptor_ids_from_set,
}
